# LangGraph: Reducers, Routing, Parallel, Loops, Memory and Human-in-the-Loop

In the earlier notebooks we built straight-line graphs: `START -> node -> node -> END` (BMI, simple LLM QA, prompt chaining) and a first tool-calling agent.
This notebook takes the next steps, one idea at a time:

1. Reducers: how a node's output is merged into the state
2. Routing: conditional edges that choose the next node
3. Branching workflow: an LLM decides which path to take
4. Parallel execution: fan-out and fan-in
5. Support workflow with a review loop: a graph that goes back and retries
6. The agent loop as a graph: LLM + tools + loop, built by hand
7. Memory: checkpointer and `thread_id`
8. Human-in-the-loop (HITL): pausing a graph and waiting for a person
9. The prebuilt way: `create_agent` + approval middleware

Each part follows the same flow: the idea from first principles, then a small piece of code, then what to notice in the output.

**One mental model for the whole notebook**

A LangGraph app has only three ingredients:

- **State**: a shared whiteboard (a Python dict) that every node can read.
- **Nodes**: plain Python functions. Each one reads the whiteboard and returns *what it wants to change*.
- **Edges**: the arrows that decide which node runs next.

Everything in this notebook (loops, parallel work, memory, human approval) is just a new way of using these three ingredients.

In [ ]:
# Install once (uncomment and run if these are missing)
# %pip install -U langgraph langchain langchain-openrouter langchain-google-genai langchain-openai python-dotenv

In [ ]:
# ===================== SETUP: run this cell first =====================
PROVIDER = "gemini"    # "openrouter", "gemini" or "openai"
MODEL = ""                 # leave empty to use the default model below

import os
from dotenv import load_dotenv

# read the .env file in this folder (override=True picks up edits without restarting the kernel)
"""if not os.path.exists(".env"):
    raise FileNotFoundError("No .env file in this folder. Create it and add your API key.")"""
load_dotenv()

if PROVIDER == "openrouter":
    KEY_NAME = "OPENROUTER_API_KEY"
    MODEL = MODEL or "google/gemini-2.5-flash-lite"
    BASE_URL = "https://openrouter.ai/api/v1"
elif PROVIDER == "gemini":
    KEY_NAME = "GOOGLE_API_KEY"
    MODEL = MODEL or "gemini-2.5-flash-lite"
    BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
elif PROVIDER == "openai":
    KEY_NAME = "OPENAI_API_KEY"
    MODEL = MODEL or "gpt-4.1-mini"
    BASE_URL = "https://api.openai.com/v1"
else:
    raise ValueError('PROVIDER must be "openrouter", "gemini" or "openai"')

API_KEY = os.getenv(KEY_NAME)
if not API_KEY:
    raise ValueError(f"{KEY_NAME} is missing. Add it to the .env file in this folder.")


def get_chat_model(temperature=0, **kwargs):
    """LangChain chat model for the chosen provider."""
    if PROVIDER == "openrouter":
        from langchain_openrouter import ChatOpenRouter
        return ChatOpenRouter(model=MODEL, api_key=API_KEY, base_url=BASE_URL, temperature=temperature, **kwargs)
    elif PROVIDER == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=MODEL, api_key=API_KEY, temperature=temperature, **kwargs)
    else:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=MODEL, api_key=API_KEY, base_url=BASE_URL, temperature=temperature, **kwargs)


def get_openai_client():
    """Plain OpenAI SDK client pointed at the chosen provider. Returns (client, model)."""
    from openai import OpenAI
    return OpenAI(api_key=API_KEY, base_url=BASE_URL), MODEL


def show_config():
    """Print the settings without showing the full key."""
    print(f"provider : {PROVIDER}")
    print(f"model    : {MODEL}")
    print(f"key      : {KEY_NAME} = {API_KEY[:6]}...{API_KEY[-2:]} ({len(API_KEY)} chars)")
    print(f"endpoint : {BASE_URL}")


show_config()

In [ ]:
# ---- Common imports used across the notebook ----
import operator                                   # operator.add is used as a "merge rule" (reducer) for lists
from typing import TypedDict, Annotated           # TypedDict = shape of the state, Annotated = attach a reducer to a key

from langgraph.graph import StateGraph, START, END   # the graph builder and the two special nodes
from IPython.display import Image, display


# The one model object we reuse everywhere
llm = get_chat_model()


def show_graph(app):
    """Draw the compiled graph. Falls back to Mermaid text if the image service is unreachable."""
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())


# Quick smoke test: if this prints a reply, the model is wired up correctly
print(llm.invoke("Say hello in five words.").content)

---
## Part 1: Reducers (how updates are merged into the state)

### First principles

A node does not get to replace the whole state. It returns a small dict of **updates**, for example `{"logs": ["step 2 done"]}`.
LangGraph then has to answer one question for every key in that update:

> The whiteboard already has a value for `logs`. A node just sent a new value. What should the whiteboard hold now?

There are only two sensible answers:

- **Overwrite**: throw away the old value, keep the new one. This is the default.
- **Combine**: merge old and new using some rule, for example "append the new list to the old list".

A **reducer** is that rule. It is just a function `reducer(old_value, new_value) -> merged_value`.
You attach it to a key using `Annotated[type, reducer]`.

**Analogy:** think of a class attendance register.
The "current topic" written on the board is overwritten every lecture (no reducer).
The attendance register is never erased; every day's entry is added below the previous one (reducer = append).

### A note on the style used in earlier notebooks

In the BMI and prompt chaining notebooks, the nodes changed `state` directly and returned the whole thing. That works for simple keys.
From here on, nodes return **only the keys they change**. This matters once reducers come in: if a key has an "append" reducer and you return the whole state, the old list is appended to itself and you get duplicates.

### 1a. Without a reducer: the last write wins

In [ ]:
class OverwriteState(TypedDict):
    logs: list[str]          # no reducer attached -> every update overwrites the old value


def step_1(state: OverwriteState):
    # return ONLY the key we want to change
    return {"logs": ["step 1 done"]}


def step_2(state: OverwriteState):
    return {"logs": ["step 2 done"]}


builder = StateGraph(OverwriteState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_edge(START, "step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", END)
app = builder.compile()

print(app.invoke({"logs": ["started"]}))

We started with `["started"]`, step 1 wrote its list, step 2 wrote its list. Only `["step 2 done"]` survives. The history is lost.

### 1b. With a reducer: `operator.add` appends lists

`operator.add(a, b)` is the same as `a + b`. For two lists, `+` joins them: `["a"] + ["b"] == ["a", "b"]`.
So `Annotated[list[str], operator.add]` means: "when a node sends a new list for this key, join it to the end of the old list".

In [ ]:
class AppendState(TypedDict):
    # Annotated[ <type>, <reducer> ]
    # reducer = operator.add  ->  merged = old_list + new_list
    logs: Annotated[list[str], operator.add]


builder = StateGraph(AppendState)
builder.add_node("step_1", step_1)       # same node functions as before
builder.add_node("step_2", step_2)
builder.add_edge(START, "step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", END)
app = builder.compile()

print(app.invoke({"logs": ["started"]}))

Same nodes, same edges. The only change is one line in the state definition, and now the full history is kept.

### 1c. Writing your own reducer

A reducer can be any function that takes `(old, new)` and returns the merged value. Example: we only want to remember the **highest** score ever seen.

In [ ]:
def keep_max(old: int, new: int) -> int:
    """Custom reducer: whichever value is bigger stays on the whiteboard."""
    return max(old, new)


class ScoreState(TypedDict):
    best_score: Annotated[int, keep_max]


def attempt_1(state: ScoreState):
    return {"best_score": 72}

def attempt_2(state: ScoreState):
    return {"best_score": 65}     # lower than 72, so the reducer ignores it


builder = StateGraph(ScoreState)
builder.add_node("attempt_1", attempt_1)
builder.add_node("attempt_2", attempt_2)
builder.add_edge(START, "attempt_1")
builder.add_edge("attempt_1", "attempt_2")
builder.add_edge("attempt_2", END)
app = builder.compile()

print(app.invoke({"best_score": 0}))

### 1d. `add_messages`: the reducer for chat history

Chat apps keep a list of messages. LangGraph ships a ready-made reducer for this, `add_messages`, which you already saw in the tool-calling notebook.
It appends new messages like `operator.add`, with two extras:

- it accepts plain dicts like `{"role": "user", "content": "hi"}` and converts them to message objects
- if a new message has the same `id` as an existing one, it **replaces** it instead of adding a duplicate

LangGraph also ships `MessagesState`, which is simply a state with one key: `messages: Annotated[list, add_messages]`. We use it from Part 6 onwards.

In [ ]:
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

old = [HumanMessage(content="Hi, I am Riya")]
new = [AIMessage(content="Hello Riya!")]

merged = add_messages(old, new)      # a reducer is just a function, so we can call it directly
for m in merged:
    print(type(m).__name__, "->", m.content)

---
## Part 2: Routing with conditional edges

### First principles

So far every edge was fixed: after `step_1` always comes `step_2`. Real workflows need decisions: *if* the marks are high, do this, *else* do that.

A **conditional edge** replaces a fixed arrow with a small function called a **router**:

- the router receives the current state
- it returns the **name of the next node** (a string)
- it must not change the state; it only looks and points

**Analogy:** a traffic police officer at a junction. He does not drive your car or change what is in it. He looks at your car and points left or right.

The API:

```python
builder.add_conditional_edges(source_node, router_function, [list of possible next nodes])
```

The last argument (the list of possible destinations) is optional for running, but it lets LangGraph **draw** the diagram correctly.
Without it, the drawing cannot know where the router might go. That is why, in the tool-calling notebook, the `tools` node appeared floating and disconnected in the diagram.

This first example has no LLM at all, so the routing idea stays clear.

In [ ]:
class ResultState(TypedDict):
    marks: int
    message: str


# ---- Router: looks at the state and returns the NAME of the next node ----
def route_by_marks(state: ResultState) -> str:
    if state["marks"] >= 75:
        return "distinction"
    elif state["marks"] >= 40:
        return "passed"
    return "failed"


# ---- The three possible destinations ----
def distinction(state: ResultState):
    return {"message": "Distinction! Excellent work."}

def passed(state: ResultState):
    return {"message": "Passed. Good job."}

def failed(state: ResultState):
    return {"message": "Failed. Let's plan a re-attempt."}


builder = StateGraph(ResultState)
builder.add_node("distinction", distinction)
builder.add_node("passed", passed)
builder.add_node("failed", failed)

# Conditional edge straight from START: the very first decision is made by the router
builder.add_conditional_edges(START, route_by_marks, ["distinction", "passed", "failed"])

# All three paths end the graph
builder.add_edge("distinction", END)
builder.add_edge("passed", END)
builder.add_edge("failed", END)

app = builder.compile()
show_graph(app)

In [ ]:
for m in [88, 55, 23]:
    result = app.invoke({"marks": m})
    print(m, "->", result["message"])

The dotted arrows in the diagram are conditional edges. Only one of them is taken on each run, depending on what the router returns.

---
## Part 3: Branching workflow (the LLM decides the branch)

### First principles

In Part 2 the decision was a simple `if`. Many real decisions are fuzzy: "is this customer review happy or angry?" There is no clean rule for that, but an LLM is good at it.

So we split the work into two clear jobs:

1. A **classifier node** asks the LLM a narrow question and **writes the answer into the state** (`sentiment`).
2. A **router** reads that value and picks the branch.

Why not let the router call the LLM itself? Because routers should only look and point. Putting the LLM call in a node means the answer is saved in the state, so we can print it, debug it and reuse it later.

**Analogy:** a hospital reception. The receptionist (classifier node) asks a few questions and writes "cardiology" on your slip. The signboard (router) reads the slip and sends you to the right floor.

Use case: replying to customer reviews. Positive reviews get a thank-you, negative reviews get an apology and a fix.

In [ ]:
class ReviewState(TypedDict):
    review: str
    sentiment: str     # filled by the classifier node
    reply: str         # filled by whichever branch runs


# ---- Node 1: ask the LLM a narrow question, store the answer in the state ----
def classify_sentiment(state: ReviewState):
    prompt = (
        "Classify the sentiment of this customer review. "
        "Answer with exactly one word: positive or negative.\n\n"
        f"Review: {state['review']}"
    )
    answer = llm.invoke(prompt).content.strip().lower()

    # LLMs sometimes add punctuation or extra words, so we normalise the answer
    sentiment = "negative" if "negative" in answer else "positive"
    return {"sentiment": sentiment}


# ---- Router: reads the label and points to a branch ----
def route_by_sentiment(state: ReviewState) -> str:
    return "positive_reply" if state["sentiment"] == "positive" else "negative_reply"


# ---- Branch A ----
def positive_reply(state: ReviewState):
    prompt = f"Write a short, warm thank-you reply (2 sentences) to this review:\n{state['review']}"
    return {"reply": llm.invoke(prompt).content}


# ---- Branch B ----
def negative_reply(state: ReviewState):
    prompt = (
        "Write a short, polite reply (3 sentences) to this unhappy customer. "
        "Apologise, address the specific problem, and offer a next step.\n"
        f"Review: {state['review']}"
    )
    return {"reply": llm.invoke(prompt).content}


builder = StateGraph(ReviewState)
builder.add_node("classify_sentiment", classify_sentiment)
builder.add_node("positive_reply", positive_reply)
builder.add_node("negative_reply", negative_reply)

builder.add_edge(START, "classify_sentiment")
builder.add_conditional_edges("classify_sentiment", route_by_sentiment, ["positive_reply", "negative_reply"])
builder.add_edge("positive_reply", END)
builder.add_edge("negative_reply", END)

review_app = builder.compile()
show_graph(review_app)

In [ ]:
reviews = [
    "The course was amazing, the trainer explained LangGraph really clearly!",
    "The app crashes every time I try to upload my assignment. Very frustrating.",
]

for r in reviews:
    result = review_app.invoke({"review": r})
    print("REVIEW   :", r)
    print("SENTIMENT:", result["sentiment"])
    print("REPLY    :", result["reply"])
    print("-" * 80)

Notice the shape: **classify -> route -> specialised handler**. This same shape is used for support ticket triage, intent detection in chatbots, and choosing which expert prompt to use.

---
## Part 4: Parallel execution (fan-out and fan-in)

### First principles

If two tasks do not depend on each other, there is no reason to wait for one before starting the other.

- **Fan-out**: draw more than one edge out of the same node. All the targets run **at the same time**.
- **Fan-in**: draw edges from all those nodes into one node. That node waits until **all** of them have finished, then runs once.

**Analogy:** cooking dinner. Rice, dal and sabzi are cooked on three burners at once (fan-out). You serve the thali only when all three are ready (fan-in).

**How LangGraph runs this: supersteps.** LangGraph runs the graph in rounds called *supersteps*. In each round it runs every node that is ready, together, then applies all their updates to the state, then moves to the next round. Parallel nodes are simply nodes that land in the same round.

**Why reducers matter here.** If three nodes in the same round all write to the same key, LangGraph receives three updates at once. Without a reducer it cannot know which to keep, so it raises an error instead of guessing. With a reducer like `operator.add`, it simply combines all three.

Let's see that error first, with no LLM involved.

In [ ]:
class NoReducerState(TypedDict):
    result: str                 # no reducer!


def worker_a(state): return {"result": "A"}
def worker_b(state): return {"result": "B"}


builder = StateGraph(NoReducerState)
builder.add_node("worker_a", worker_a)
builder.add_node("worker_b", worker_b)
builder.add_edge(START, "worker_a")      # two edges out of START = fan-out
builder.add_edge(START, "worker_b")
builder.add_edge("worker_a", END)
builder.add_edge("worker_b", END)
bad_app = builder.compile()

try:
    bad_app.invoke({"result": ""})
except Exception as e:
    print(type(e).__name__, "->", str(e)[:200])

`InvalidUpdateError`: two nodes wrote to `result` in the same superstep and there was no rule to combine them.

### The real example: a study card generator

Given a topic, three LLM calls run in parallel:

- a simple definition
- a real-life analogy
- one interview question

Each writes into the same `notes` list (made safe with `operator.add`), and a final `combine` node turns them into a single study card.

In [ ]:
class StudyCardState(TypedDict):
    topic: str
    notes: Annotated[list[str], operator.add]   # three parallel nodes append here safely
    card: str


def write_definition(state: StudyCardState):
    text = llm.invoke(f"Define '{state['topic']}' in 2 simple sentences for a beginner.").content
    return {"notes": [f"DEFINITION:\n{text}"]}


def write_analogy(state: StudyCardState):
    text = llm.invoke(f"Give one everyday analogy (2 sentences) that explains '{state['topic']}'.").content
    return {"notes": [f"ANALOGY:\n{text}"]}


def write_question(state: StudyCardState):
    text = llm.invoke(f"Write one interview question (with a 2-line answer) about '{state['topic']}'.").content
    return {"notes": [f"INTERVIEW QUESTION:\n{text}"]}


# ---- Fan-in node: runs once, after ALL three writers have finished ----
def combine(state: StudyCardState):
    card = f"STUDY CARD: {state['topic']}\n\n" + "\n\n".join(state["notes"])
    return {"card": card}


builder = StateGraph(StudyCardState)
builder.add_node("write_definition", write_definition)
builder.add_node("write_analogy", write_analogy)
builder.add_node("write_question", write_question)
builder.add_node("combine", combine)

# fan-out: three edges leave START
builder.add_edge(START, "write_definition")
builder.add_edge(START, "write_analogy")
builder.add_edge(START, "write_question")

# fan-in: three edges arrive at combine
builder.add_edge("write_definition", "combine")
builder.add_edge("write_analogy", "combine")
builder.add_edge("write_question", "combine")

builder.add_edge("combine", END)

card_app = builder.compile()
show_graph(card_app)

In [ ]:
result = card_app.invoke({"topic": "Reducers in LangGraph"})
print(result["card"])

Things to notice:

- The order of the three notes can change between runs. They finish in whatever order the LLM responds, and `operator.add` appends them as they arrive.
- `combine` ran only once, after all three were done. That is fan-in.
- Total time is roughly the time of the **slowest** call, not the sum of all three.

---
## Part 5: Support workflow with a review loop

### First principles

A **loop** in a graph is nothing special: it is just an edge that points **backwards** to a node that already ran.

The pattern here is called **generate and review** (also called evaluator-optimizer):

1. A writer node drafts a reply.
2. A reviewer node checks it against a checklist.
3. A router decides: approved? finish. Not approved? go back to the writer **with the feedback**.

**Analogy:** submitting an assignment. The teacher returns it with red-pen comments, you fix those exact points and resubmit, until it is accepted.

**Every loop needs an exit.** LLM reviewers can be fussy and may never say "approved". So we always add a maximum number of attempts.
LangGraph also has its own safety net: a `recursion_limit` (default 25 steps per run). If a graph keeps looping past that, it stops with a `GraphRecursionError`.

Use case: a customer support reply that must follow company rules before it is sent.

In [ ]:
MAX_ATTEMPTS = 3


class SupportState(TypedDict):
    ticket: str        # the customer's message
    draft: str         # current reply draft
    feedback: str      # reviewer's comments ("" on the first attempt)
    approved: bool
    attempts: int


# ---- Writer: uses the feedback if there is any ----
def write_draft(state: SupportState):
    prompt = f"You are a customer support agent. Write a reply to this ticket:\n{state['ticket']}"
    if state.get("feedback"):
        prompt += (
            f"\n\nYour previous draft was:\n{state['draft']}"
            f"\n\nA reviewer asked for these fixes:\n{state['feedback']}\nRewrite the reply."
        )
    draft = llm.invoke(prompt).content
    return {"draft": draft, "attempts": state.get("attempts", 0) + 1}


# ---- Reviewer: a second LLM call with a strict checklist ----
def review_draft(state: SupportState):
    prompt = (
        "You review customer support replies. Check this reply against the rules:\n"
        "1. It apologises for the problem.\n"
        "2. It gives a clear next step with a timeline.\n"
        "3. It is under 80 words.\n"
        "4. It does not promise a refund.\n\n"
        f"Reply:\n{state['draft']}\n\n"
        "If ALL rules are met, answer with exactly the word APPROVED. "
        "Otherwise list only what must be fixed, in at most two short lines."
    )
    verdict = llm.invoke(prompt).content.strip()
    approved = verdict.upper().startswith("APPROVED")
    return {"approved": approved, "feedback": "" if approved else verdict}


# ---- Router: the exit condition of the loop ----
def after_review(state: SupportState) -> str:
    if state["approved"] or state["attempts"] >= MAX_ATTEMPTS:
        return "send_reply"
    return "write_draft"          # <- the backward edge: this is the loop


def send_reply(state: SupportState):
    # In a real system this would call an email / ticketing API
    status = "approved" if state["approved"] else "sent after max attempts, flag for a human"
    print(f"Reply sent ({status}).")
    return {}


builder = StateGraph(SupportState)
builder.add_node("write_draft", write_draft)
builder.add_node("review_draft", review_draft)
builder.add_node("send_reply", send_reply)

builder.add_edge(START, "write_draft")
builder.add_edge("write_draft", "review_draft")
builder.add_conditional_edges("review_draft", after_review, ["write_draft", "send_reply"])
builder.add_edge("send_reply", END)

support_app = builder.compile()
show_graph(support_app)

To watch the loop happen, we use `stream(..., stream_mode="updates")` instead of `invoke`.
`invoke` returns only the final state. `stream` yields one item **per node as it finishes**, so we can see each attempt.

In [ ]:
ticket = "I paid for the premium course 3 days ago but still can't access the videos. This is ridiculous."

start_state = {"ticket": ticket, "draft": "", "feedback": "", "approved": False, "attempts": 0}

for update in support_app.stream(start_state, stream_mode="updates"):
    for node_name, changes in update.items():
        print(f"=== {node_name} ===")
        if node_name == "write_draft":
            print(f"(attempt {changes['attempts']})\n{changes['draft']}\n")
        elif node_name == "review_draft":
            print("APPROVED" if changes["approved"] else f"Feedback: {changes['feedback']}\n")

---
## Part 6: The agent loop as a graph

### First principles

What makes something an "agent"? Strip away the hype and it is this:

> **An LLM in a loop, with tools, that decides for itself when it is done.**

Important fact: **the LLM cannot run code.** It only produces text. When we "give it tools", we send it a description of each tool (name, what it does, what arguments it takes). If it wants a tool, it replies with a structured request, a **tool call**: "please run `multiply` with `a=23, b=17`". Our code runs the tool and sends the result back as a `ToolMessage`. Then the LLM looks again and decides what to do next.

So the message history of one question looks like this:

| Step | Message | Who creates it |
|---|---|---|
| 1 | `HumanMessage`: "What is 23 x 17, plus 100?" | user |
| 2 | `AIMessage` with `tool_calls=[multiply(23, 17)]` | LLM |
| 3 | `ToolMessage`: "391" | our tool node |
| 4 | `AIMessage` with `tool_calls=[add(391, 100)]` | LLM |
| 5 | `ToolMessage`: "491" | our tool node |
| 6 | `AIMessage`: "The answer is 491." (no tool calls) | LLM, so the loop stops |

This is exactly the review loop from Part 5, with different roles:

- `agent` node: call the LLM (writer)
- router: "did the LLM ask for a tool?" yes -> `tools`, no -> `END`
- `tools` node: run the requested tools, then go **back** to `agent`

**Analogy:** a manager (LLM) and an assistant (tool node). The manager cannot do calculations himself, he only writes notes: "calculate 23 x 17". The assistant does it and hands the result back. The manager reads it and either writes another note or gives the final answer.

In [ ]:
from langchain_core.tools import tool                 # turns a Python function into a tool the LLM can request
from langgraph.graph import MessagesState             # ready-made state: {"messages": Annotated[list, add_messages]}
from langgraph.prebuilt import ToolNode               # a node that runs whatever tool calls the last AIMessage asked for


# ---- Tools: the docstring and type hints become the description the LLM reads ----
@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


tools = [add, multiply]

# bind_tools sends the tool descriptions to the LLM with every call
llm_with_tools = llm.bind_tools(tools)


# ---- Node 1: the "brain" ----
def call_model(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}          # add_messages appends it to the history


# ---- Router: did the LLM ask for a tool? ----
def should_continue(state: MessagesState) -> str:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END


# ---- Node 2: the "hands" (prebuilt: reads tool_calls, runs them, returns ToolMessages) ----
tool_node = ToolNode(tools)


builder = StateGraph(MessagesState)
builder.add_node("agent", call_model)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, ["tools", END])   # the list fixes the "floating tools node" drawing
builder.add_edge("tools", "agent")                                          # the loop back

agent_app = builder.compile()
show_graph(agent_app)

In [ ]:
result = agent_app.invoke(
    {"messages": [{"role": "user", "content": "What is 23 multiplied by 17, then add 100 to it? Use the tools."}]}
)

# pretty_print shows each message with its type, so we can see the whole loop
for msg in result["messages"]:
    msg.pretty_print()

Read the output top to bottom and match it against the table above. Every `Tool Calls` block is one trip around the loop.

**Shortcut:** LangGraph ships this router as `tools_condition` (`from langgraph.prebuilt import tools_condition`). It does exactly what `should_continue` does. We wrote our own so the logic is visible.

---
## Part 7: Memory with a checkpointer and `thread_id`

### First principles

Every call to `invoke` starts from a **blank whiteboard**. When the run ends, the state is thrown away. So by default a graph has no memory between calls. Let's prove it.

In [ ]:
def chatbot(state: MessagesState):
    return {"messages": [llm.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

no_memory_app = builder.compile()        # no checkpointer

no_memory_app.invoke({"messages": [{"role": "user", "content": "Hi, my name is Riya."}]})
result = no_memory_app.invoke({"messages": [{"role": "user", "content": "What is my name?"}]})
print(result["messages"][-1].content)

It does not know. The second call never saw the first.

### The fix: a checkpointer

A **checkpointer** saves a snapshot of the state after every step, and loads it back at the start of the next call.

But one app serves many users and many conversations. Whose snapshot should it load? That is what the **`thread_id`** is for. It is a label for one conversation. Same `thread_id` = continue that conversation. New `thread_id` = fresh start.

**Analogy:** WhatsApp. The app (graph) is the same for everyone. Each chat window (thread) has its own history. Opening a chat with a different person shows a different history.

Or think of a video game: the checkpointer is the save system, the `thread_id` is the save slot number.

We pass the `thread_id` through a `config` dict:

```python
config = {"configurable": {"thread_id": "riya-chat"}}
```

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver   # stores snapshots in RAM (good for learning and demos)

checkpointer = InMemorySaver()
memory_app = builder.compile(checkpointer=checkpointer)  # same graph, now with a save system


config_riya = {"configurable": {"thread_id": "riya-chat"}}

memory_app.invoke({"messages": [{"role": "user", "content": "Hi, my name is Riya."}]}, config_riya)
result = memory_app.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config_riya)
print("Same thread :", result["messages"][-1].content)


# A different thread_id is a different conversation: it starts empty
config_other = {"configurable": {"thread_id": "someone-else"}}
result = memory_app.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config_other)
print("New thread  :", result["messages"][-1].content)

Notice that on the second call we only sent the new question. The checkpointer loaded the earlier messages, and the `add_messages` reducer appended the new one to them. **Reducers and memory work together.**

We can also look inside a saved thread with `get_state`:

In [ ]:
snapshot = memory_app.get_state(config_riya)

print("Messages saved in this thread:", len(snapshot.values["messages"]))
for m in snapshot.values["messages"]:
    print(f"  {m.type:>5}: {m.content[:70]}")

**About `InMemorySaver`:** it lives in RAM, so it is wiped when the kernel restarts. In production you swap it for a database-backed checkpointer (for example `SqliteSaver` or `PostgresSaver`). The graph code does not change, only the checkpointer object.

---
## Part 8: Human-in-the-loop (HITL) with `interrupt`

### First principles

Some actions should not happen without a person saying yes: sending an email to a customer, issuing a refund, deleting data.
So we need a graph that can **pause in the middle**, show something to a human, **wait** (seconds or days), and then **continue from the same point**.

How can a program wait for days? It cannot keep a Python function running that long. The trick:

1. **Pause** = save the state with the checkpointer, then stop the run.
2. **Resume** = load the saved state for that `thread_id` and continue.

That is why **HITL always needs a checkpointer and a `thread_id`.** Part 7 was the foundation for this part.

The two tools:

- `interrupt(value)`: called inside a node. The first time, it stops the graph and hands `value` to the caller (the question for the human).
- `Command(resume=answer)`: what we pass to `invoke` to continue. Inside the node, `interrupt(...)` now **returns** `answer`.

**Analogy:** a bank loan file. The clerk prepares everything, then the file goes into the manager's tray (pause, saved). The manager may sign it tomorrow. When he does, the clerk picks up the same file and continues from where it stopped.

**One gotcha:** when the graph resumes, the paused node runs again **from its first line**. Code before `interrupt()` runs twice. So put side effects (sending emails, writing to databases) **after** the interrupt or in a later node, never before it.

Use case: a refund email that a support manager must approve.

In [ ]:
from langgraph.types import interrupt, Command


class RefundState(TypedDict):
    complaint: str
    draft_email: str
    decision: str          # "approve" or "reject", filled by the human


# ---- Node 1: the LLM drafts the email ----
def draft_refund_email(state: RefundState):
    prompt = f"Write a short, polite email (under 70 words) confirming a refund for this complaint:\n{state['complaint']}"
    return {"draft_email": llm.invoke(prompt).content}


# ---- Node 2: pause and ask the human ----
def human_approval(state: RefundState):
    # First run: the graph STOPS here and the dict below is shown to the caller.
    # After resume: interrupt() RETURNS whatever was passed in Command(resume=...).
    decision = interrupt({
        "question": "Send this refund email? (approve / reject)",
        "draft_email": state["draft_email"],
    })
    return {"decision": decision}


def route_after_approval(state: RefundState) -> str:
    return "send_email" if state["decision"] == "approve" else "cancel"


# ---- Side effect lives in its own node, AFTER the approval ----
def send_email(state: RefundState):
    print("EMAIL SENT:\n", state["draft_email"])
    return {}


def cancel(state: RefundState):
    print("Email cancelled by the reviewer.")
    return {}


builder = StateGraph(RefundState)
builder.add_node("draft_refund_email", draft_refund_email)
builder.add_node("human_approval", human_approval)
builder.add_node("send_email", send_email)
builder.add_node("cancel", cancel)

builder.add_edge(START, "draft_refund_email")
builder.add_edge("draft_refund_email", "human_approval")
builder.add_conditional_edges("human_approval", route_after_approval, ["send_email", "cancel"])
builder.add_edge("send_email", END)
builder.add_edge("cancel", END)

refund_app = builder.compile(checkpointer=InMemorySaver())   # no checkpointer = no pausing
show_graph(refund_app)

### Step 1: run until the graph pauses

In [ ]:
config = {"configurable": {"thread_id": "refund-101"}}

result = refund_app.invoke(
    {"complaint": "Order #101: the headphones arrived broken. I want my money back."},
    config,
)

# When a graph pauses, the result contains an "__interrupt__" entry with what interrupt() sent out
request = result["__interrupt__"][0].value
print(request["question"])
print("-" * 60)
print(request["draft_email"])

In [ ]:
# The graph is paused. get_state shows which node is waiting to continue.
print("Waiting at:", refund_app.get_state(config).next)

### Step 2: the human decides, and we resume

In a real app this answer comes from a web UI, Slack button, etc. Here we set it in a variable.
Try it once with `"approve"` and once with `"reject"` (use a new `thread_id` for the second try, or re-run Step 1).

In [ ]:
human_answer = "approve"          # or "reject"   (you could also use: human_answer = input("approve/reject: "))

final = refund_app.invoke(Command(resume=human_answer), config)   # same config = same paused thread
print("\nDecision recorded in state:", final["decision"])

---
## Part 9: The prebuilt way: `create_agent` + approval middleware

### First principles

In Parts 6, 7 and 8 we built three things by hand: the agent loop, memory, and a human approval pause. That was to understand the machinery.
In daily work you rarely build the agent loop yourself. LangChain's `create_agent` builds **the same graph** for you: a model node, a tools node, and the loop between them.

What about adding behaviour like "ask a human before running risky tools"? Instead of editing the graph, you attach **middleware**.

**Middleware** = code that runs *around* the agent's steps (before the model is called, after the model replies, around a tool call). It can inspect, change or pause what is happening, without you touching the loop.

**Analogy:** a security guard at an office gate. Employees (the agent loop) go about their work as usual. The guard checks only certain things, for example "anyone carrying company laptops out needs a manager's signature".

`HumanInTheLoopMiddleware` is that guard for tools:

- After the model replies, it looks at the tool calls.
- If a tool is listed in `interrupt_on`, it **pauses** the graph (using the same `interrupt` from Part 8) before that tool runs.
- Safe tools run without asking.

When resuming, the human sends a list of **decisions**, one per paused tool call:

| Decision | Meaning |
|---|---|
| `{"type": "approve"}` | run the tool exactly as the model asked |
| `{"type": "edit", "edited_action": {"name": ..., "args": {...}}}` | run it, but with changed arguments |
| `{"type": "reject", "message": "reason"}` | do not run it; the reason is sent back to the model |

Use case: a support agent that can freely look up orders, but needs approval before issuing a refund.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware


# ---- Tools (fake data so the notebook runs anywhere) ----
@tool
def get_order_status(order_id: str) -> str:
    """Look up the delivery status of an order."""
    fake_db = {"1023": "Delivered on 12 Nov, customer reported damage", "1024": "In transit"}
    return fake_db.get(order_id, "Order not found")


@tool
def issue_refund(order_id: str, amount: float) -> str:
    """Issue a refund to the customer. This moves real money."""
    return f"Refund of Rs {amount} issued for order {order_id}."


support_agent = create_agent(
    model=llm,
    tools=[get_order_status, issue_refund],
    system_prompt="You are a support agent. Check the order status before issuing any refund.",
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "issue_refund": True,          # pause and ask a human before this tool runs
                "get_order_status": False,     # safe, read-only: runs without asking
            }
        )
    ],
    checkpointer=InMemorySaver(),              # required: pausing = saving state
)

show_graph(support_agent)

Look at the diagram: it is the same loop as Part 6 (`model` = our `agent` node, `tools` = our `tools` node).
The only new box is `HumanInTheLoopMiddleware.after_model`. That is the security guard, placed between the model's reply and the tools.

### Step 1: run until the middleware pauses

In [ ]:
config = {"configurable": {"thread_id": "agent-1023"}}

result = support_agent.invoke(
    {"messages": [{"role": "user", "content": "Order 1023 arrived broken. Please refund Rs 499."}]},
    config,
)

if "__interrupt__" in result:
    hitl_request = result["__interrupt__"][0].value
    for action in hitl_request["action_requests"]:
        print("Tool waiting for approval:", action["name"])
        print("Arguments                :", action["args"])
else:
    print("No approval needed:", result["messages"][-1].content)

### Step 2: approve and resume

There must be one decision per tool call in `action_requests`, in the same order.

In [ ]:
decisions = [{"type": "approve"} for _ in hitl_request["action_requests"]]

result = support_agent.invoke(Command(resume={"decisions": decisions}), config)

for msg in result["messages"]:
    msg.pretty_print()

### Same thing, but rejected

New thread, same request. This time the reviewer rejects with a reason. The tool never runs; the reason goes back to the model, which replies to the customer accordingly.

In [ ]:
config_2 = {"configurable": {"thread_id": "agent-1023-rejected"}}

result = support_agent.invoke(
    {"messages": [{"role": "user", "content": "Order 1023 arrived broken. Please refund Rs 499."}]},
    config_2,
)

if "__interrupt__" in result:
    pending = result["__interrupt__"][0].value["action_requests"]
    decisions = [
        {"type": "reject", "message": "Refunds above Rs 300 need a photo of the damage first. Ask the customer for it."}
        for _ in pending
    ]
    result = support_agent.invoke(Command(resume={"decisions": decisions}), config_2)

print(result["messages"][-1].content)

### Manual vs prebuilt: same ideas, less code

| What we built by hand | Prebuilt equivalent |
|---|---|
| `agent` node + `tools` node + `should_continue` loop (Part 6) | `create_agent(model, tools)` |
| `compile(checkpointer=...)` + `thread_id` (Part 7) | `create_agent(..., checkpointer=...)` + the same `thread_id` config |
| `interrupt()` in a node + `Command(resume=...)` (Part 8) | `HumanInTheLoopMiddleware(interrupt_on={...})` + `Command(resume={"decisions": [...]})` |

Build by hand when the flow is your own design (branching, parallel steps, review loops). Use `create_agent` when what you need is a standard tool-using agent.

---
## Recap

| Concept | One-line idea | API |
|---|---|---|
| Reducer | the rule for merging a node's update into the state | `Annotated[list, operator.add]`, `add_messages` |
| Routing | a function that looks at the state and names the next node | `add_conditional_edges(src, router, [targets])` |
| Branching | classify in a node, route on the stored label | classifier node + router |
| Parallel | many edges out (fan-out), many edges in (fan-in) | multiple `add_edge` + a reducer on shared keys |
| Loop | an edge pointing back, plus an exit condition | conditional edge back to an earlier node |
| Agent | LLM + tools + loop until no tool calls | `bind_tools`, `ToolNode`, `tools_condition` |
| Memory | save state per conversation | `compile(checkpointer=InMemorySaver())` + `thread_id` |
| HITL | pause, wait for a human, resume | `interrupt()`, `Command(resume=...)` |
| Prebuilt agent | the agent loop, ready-made, extended by middleware | `create_agent`, `HumanInTheLoopMiddleware` |

## Practice exercises

1. **Reducers:** add a `history` key with `operator.add` to the BMI workflow, so each node logs what it did.
2. **Routing:** extend the review classifier with a third label, `neutral`, and a third branch.
3. **Parallel:** add a fourth writer to the study card graph that lists "common mistakes". Did you need to change `combine`?
4. **Loop:** in the support workflow, set `MAX_ATTEMPTS = 1` and make the reviewer stricter. What happens?
5. **Agent:** add a `get_weather(city)` tool with fake data and ask a question that needs both weather and maths.
6. **Memory:** chat on two different `thread_id`s and print `get_state` for each.
7. **HITL:** in Part 9, try an `edit` decision that changes the refund amount to 299 before it runs.